In [7]:
import sys, subprocess
def pip_install(pkg):
    try:
        __import__(pkg.split('==')[0].replace('-', '_'))
    except:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

pip_install("polars>=1.0.0")
pip_install("bottleneck>=1.3.7")
pip_install("pandas>=2.0.0")

import polars as pl
import pandas as pd
import bottleneck as bn
from typing import Dict

# ---------------------------------------------------------
#                   ЧАСТЬ 1. POLARS
# ---------------------------------------------------------

TRAIN_PATHS = ["/content/train.csv", "/mnt/data/train.csv", "train.csv"]
for _p in TRAIN_PATHS:
    try:
        _ = open(_p, "rb").close()
        TRAIN_PATH = _p
        break
    except:
        TRAIN_PATH = None

if TRAIN_PATH is None:
    raise FileNotFoundError("Не найден train.csv. Поместите файл в /content или /mnt/data, либо загрузите через диалог.")

print(f"\n[Polars] Чтение Titanic из: {TRAIN_PATH}")
titanic_pl = pl.read_csv(TRAIN_PATH)

print("\n=== [Polars] dtypes ===")
print(titanic_pl.dtypes)

print("\n=== [Polars] describe() ===")
print(titanic_pl.describe())

print("\n=== [Polars] null_count() ===")
print(titanic_pl.null_count())

numeric_cols = [c for c, dt in zip(titanic_pl.columns, titanic_pl.dtypes) if dt in pl.NUMERIC_DTYPES]
if numeric_cols:
    print("\n=== [Polars] mean() для числовых столбцов ===")
    print(titanic_pl.select([pl.col(numeric_cols).mean()]))

print("\n=== [Polars] Кол-во пассажиров по Pclass ===")
if "Pclass" in titanic_pl.columns:
    pclass_counts = titanic_pl.get_column("Pclass").value_counts().sort("Pclass")
    print(pclass_counts)
else:
    print("Столбец Pclass не найден.")

print("\n=== [Polars] Кол-во выживших по полу (Sex) ===")
if {"Sex", "Survived"}.issubset(set(titanic_pl.columns)):
    survived_by_sex = (
        titanic_pl
        .group_by("Sex")
        .agg(pl.col("Survived").sum().alias("Survived_Count"))
        .sort("Sex")
    )
    print(survived_by_sex)
else:
    print("Не найдены столбцы Sex/Survived.")

print("\n=== [Polars] Пассажиры Age > 44 ===")
if "Age" in titanic_pl.columns:
    older_44 = titanic_pl.filter(pl.col("Age") > 44)
    print(older_44.head(10))
else:
    print("Столбец Age не найден.")


[Polars] Чтение Titanic из: /content/train.csv

=== [Polars] dtypes ===
[Int64, Int64, Int64, String, String, Float64, Int64, Int64, String, Float64, String, String]

=== [Polars] describe() ===
shape: (9, 13)
┌────────────┬─────────────┬──────────┬──────────┬───┬───────────┬───────────┬───────┬──────────┐
│ statistic  ┆ PassengerId ┆ Survived ┆ Pclass   ┆ … ┆ Ticket    ┆ Fare      ┆ Cabin ┆ Embarked │
│ ---        ┆ ---         ┆ ---      ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---   ┆ ---      │
│ str        ┆ f64         ┆ f64      ┆ f64      ┆   ┆ str       ┆ f64       ┆ str   ┆ str      │
╞════════════╪═════════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════╪══════════╡
│ count      ┆ 891.0       ┆ 891.0    ┆ 891.0    ┆ … ┆ 891       ┆ 891.0     ┆ 204   ┆ 889      │
│ null_count ┆ 0.0         ┆ 0.0      ┆ 0.0      ┆ … ┆ 0         ┆ 0.0       ┆ 687   ┆ 2        │
│ mean       ┆ 446.0       ┆ 0.383838 ┆ 2.308642 ┆ … ┆ null      ┆ 32.204208 ┆ null  ┆ null     │
│ std

/tmp/ipython-input-897667512.py:45: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  numeric_cols = [c for c, dt in zip(titanic_pl.columns, titanic_pl.dtypes) if dt in pl.NUMERIC_DTYPES]


In [8]:
# ---------------------------------------------------------
#              ЧАСТЬ 2. УСКОРЕНИЕ С PANDAS
# ---------------------------------------------------------

print(f"\n[Pandas] Чтение Titanic из: {TRAIN_PATH}")
titanic_pd = pd.read_csv(TRAIN_PATH)

if "Age" in titanic_pd.columns:
    age_values = titanic_pd["Age"].to_numpy(dtype="float64", copy=False)
    age_mean = bn.nanmean(age_values)
    age_std  = bn.nanstd(age_values)
    print("\n=== [Bottleneck] Age: nanmean и nanstd ===")
    print(f"nanmean(Age) = {age_mean:.4f}")
    print(f"nanstd(Age)  = {age_std:.4f}")
else:
    print("Столбец Age не найден в pandas-версии.")

if "Fare" in titanic_pd.columns:

    fare_new = []
    for row in titanic_pd[["Fare"]].itertuples(index=False):
        f = row.Fare
        fare_new.append(f * 1.3 if pd.notna(f) else pd.NA)
    titanic_pd["Fare_new"] = fare_new

    print("\n=== Пример столбцов Fare и Fare_new ===")
    print(titanic_pd[["Fare", "Fare_new"]].head(10))
else:
    print("Столбец Fare не найден в pandas-версии.")


[Pandas] Чтение Titanic из: /content/train.csv

=== [Bottleneck] Age: nanmean и nanstd ===
nanmean(Age) = 29.6991
nanstd(Age)  = 14.5163

=== Пример столбцов Fare и Fare_new ===
      Fare  Fare_new
0   7.2500   9.42500
1  71.2833  92.66829
2   7.9250  10.30250
3  53.1000  69.03000
4   8.0500  10.46500
5   8.4583  10.99579
6  51.8625  67.42125
7  21.0750  27.39750
8  11.1333  14.47329
9  30.0708  39.09204


In [9]:
# ---------------------------------------------------------
#              ЧАСТЬ 3. Оптимизация типов
# ---------------------------------------------------------

HOUSING_PATHS = ["/content/Housing.csv", "/mnt/data/Housing.csv", "Housing.csv"]
for _p in HOUSING_PATHS:
    try:
        _ = open(_p, "rb").close()
        HOUSING_PATH = _p
        break
    except:
        HOUSING_PATH = None

if HOUSING_PATH is None:
    print("\n[Внимание] Housing.csv не найден. Загрузите файл или укажите путь, чтобы выполнить часть 3.")
else:
    print(f"\n[Pandas] Чтение Housing из: {HOUSING_PATH}")
    housing = pd.read_csv(HOUSING_PATH)

    def mem_mb(df: pd.DataFrame) -> float:
        return df.memory_usage(deep=True).sum() / (1024**2)

    mem_before = mem_mb(housing)
    print(f"\n=== Память ДО оптимизации: {mem_before:.2f} MB ===")


    def try_parse_datetime(series: pd.Series) -> pd.Series:
        try:
            parsed = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
            success_ratio = parsed.notna().mean()
            return parsed if success_ratio > 0.95 else series
        except Exception:
            return series

    def optimize_dtypes(df: pd.DataFrame) -> (pd.DataFrame, Dict[str, str]):
        df_opt = df.copy()
        plan: Dict[str, str] = {}

        for col in df_opt.columns:
            s = df_opt[col]
            orig_dtype = s.dtype

            if s.dtype == "object":
                s_dt = try_parse_datetime(s)
                if pd.api.types.is_datetime64_any_dtype(s_dt):
                    df_opt[col] = s_dt
                    plan[col] = f"{orig_dtype} -> datetime64[ns]"
                    continue

            if s.dtype == "object":
                unique_vals = set(map(lambda x: str(x).strip().lower(), s.dropna().unique().tolist()))
                if unique_vals.issubset({"true","false","0","1","yes","no"}):
                    def to_bool(x):
                        if pd.isna(x): return pd.NA
                        v = str(x).strip().lower()
                        return v in {"true","1","yes"}
                    df_opt[col] = s.map(to_bool).astype("boolean")
                    plan[col] = f"{orig_dtype} -> boolean"
                    continue

            if s.dtype == "object":
                nunique = s.nunique(dropna=True)
                if nunique > 0 and nunique / max(len(s), 1) <= 0.5:
                    df_opt[col] = s.astype("category")
                    plan[col] = f"{orig_dtype} -> category"
                    continue

            if pd.api.types.is_integer_dtype(s):
                df_opt[col] = pd.to_numeric(s, downcast="integer")
                plan[col] = f"{orig_dtype} -> {df_opt[col].dtype}"
                continue

            if pd.api.types.is_float_dtype(s):
                df_opt[col] = pd.to_numeric(s, downcast="float")
                plan[col] = f"{orig_dtype} -> {df_opt[col].dtype}"
                continue

            if col not in plan:
                plan[col] = f"{orig_dtype} -> (без изменений)"

        return df_opt, plan

    housing_opt, conversion_plan = optimize_dtypes(housing)

    mem_after = mem_mb(housing_opt)
    print(f"=== Память ПОСЛЕ оптимизации: {mem_after:.2f} MB ===")
    print(f"=== Экономия: {mem_before - mem_after:.2f} MB ({(1 - mem_after/max(mem_before, 1e-9))*100:.1f}%) ===")

    print("\n=== План преобразований типов (колонка: было -> стало) ===")
    for c, p in conversion_plan.items():
        print(f"{c}: {p}")


[Pandas] Чтение Housing из: /content/Housing.csv

=== Память ДО оптимизации: 0.22 MB ===
=== Память ПОСЛЕ оптимизации: 0.01 MB ===
=== Экономия: 0.20 MB (94.3%) ===

=== План преобразований типов (колонка: было -> стало) ===
price: int64 -> int32
area: int64 -> int16
bedrooms: int64 -> int8
bathrooms: int64 -> int8
stories: int64 -> int8
mainroad: object -> boolean
guestroom: object -> boolean
basement: object -> boolean
hotwaterheating: object -> boolean
airconditioning: object -> boolean
parking: int64 -> int8
prefarea: object -> boolean
furnishingstatus: object -> category


/tmp/ipython-input-728942389.py:29: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
/tmp/ipython-input-728942389.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
/tmp/ipython-input-728942389.py:29: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetim